In [10]:
with open('KJV-Bible.txt', 'r', encoding='utf-8') as f:
    text = f.read() 

In [13]:
#get unique characters in text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#&'(),-.0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]abcdefghijklmnopqrstuvwxyz
81


In [22]:
#tokenize characters (char->int)
str_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_str = { i:ch for i,ch in enumerate(chars) }
def encode(s):
    return [str_to_int[c] for c in s]
def decode(l):
    return ''.join([int_to_str[i] for i in l])

print(encode("test"))
print(decode(encode("test")))

[74, 59, 73, 74]
test


In [ ]:
# encode the entire text and store in Tensor
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

In [26]:
# split data into training and validation sets
n = int(0.9 * len(data))
train = data[:n]
test = data[n:]

block_size = 8
train[:block_size+1]

tensor([46, 34, 31,  1, 34, 41, 38, 51,  1])

In [27]:
x = train[:block_size]
y = train[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([46]) the target: 34
when input is tensor([46, 34]) the target: 31
when input is tensor([46, 34, 31]) the target: 1
when input is tensor([46, 34, 31,  1]) the target: 34
when input is tensor([46, 34, 31,  1, 34]) the target: 41
when input is tensor([46, 34, 31,  1, 34, 41]) the target: 38
when input is tensor([46, 34, 31,  1, 34, 41, 38]) the target: 51
when input is tensor([46, 34, 31,  1, 34, 41, 38, 51]) the target: 1


In [ ]:
batch_size = 4 # number of independent sequences to process in parallel
block_size = 8 # maximum context length

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train if split == 'train' else test
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # get x block
    x = torch.stack([data[i:i+block_size] for i in ix])
    # get y block (1 offset from x)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        # x and y batches are already offset so we dont offset them here
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[59, 68,  1, 52, 55, 57, 62, 55],
        [58,  1, 55, 68, 58,  1, 60, 69],
        [62, 59,  1, 65, 63, 68, 61,  1],
        [59,  1, 62, 69, 75, 73, 59,  9]])
targets:
torch.Size([4, 8])
tensor([[68,  1, 52, 55, 57, 62, 55, 72],
        [ 1, 55, 68, 58,  1, 60, 69, 72],
        [59,  1, 65, 63, 68, 61,  1, 73],
        [ 1, 62, 69, 75, 73, 59,  9,  1]])
-----
when input is [59] the target: 68
when input is [59, 68] the target: 1
when input is [59, 68, 1] the target: 52
when input is [59, 68, 1, 52] the target: 55
when input is [59, 68, 1, 52, 55] the target: 57
when input is [59, 68, 1, 52, 55, 57] the target: 62
when input is [59, 68, 1, 52, 55, 57, 62] the target: 55
when input is [59, 68, 1, 52, 55, 57, 62, 55] the target: 72
when input is [58] the target: 1
when input is [58, 1] the target: 55
when input is [58, 1, 55] the target: 68
when input is [58, 1, 55, 68] the target: 58
when input is [58, 1, 55, 68, 58] the target: 1
when input is [58, 1